In [2]:
import time
import jax
import jax.numpy as jnp
from jax.experimental import sparse as jsparse
import scipy.sparse as sp
import pyamg
from pyamg.gallery import poisson

jax.config.update("jax_enable_x64", True)

In [ ]:
# SpGEMM en JAX -> nnz_out = nnz_A * nnz_B
def spgemm(A, B):
    A_i, A_k, A_v = A.indices[:,0], A.indices[:,1], A.data
    B_k, B_j, B_v = B.indices[:,0], B.indices[:,1], B.data
    mask = (A_k[:, None] == B_k[None, :])
    contrib = jnp.where(mask, A_v[:, None] * B_v[None, :], 0.)
    return jnp.zeros((A.shape[0], B.shape[1])).at[A_i[:, None], B_j[None, :]].add(contrib)

# on test sur petit cas
A_sp = sp.random(20, 20, density=0.2, format='csr', dtype='float64', random_state=0)
B_sp = sp.random(20, 15, density=0.2, format='csr', dtype='float64', random_state=1)
A = jsparse.BCOO.from_scipy_sparse(A_sp)
B = jsparse.BCOO.from_scipy_sparse(B_sp)

C = spgemm(A, B)
C_ref = (A_sp @ B_sp).toarray()
print("correct:", jnp.allclose(C, C_ref))

correct: True


In [4]:
# on test sur grand cas
ml = pyamg.ruge_stuben_solver(poisson((200, 200), format='csr'), coarse_solver='jacobi')
A_sp = ml.levels[0].A.astype('float64')
P_sp = ml.levels[0].P.astype('float64')
R_sp = P_sp.T.tocsr()

A_jax = jsparse.BCOO.from_scipy_sparse(A_sp)
P_jax = jsparse.BCOO.from_scipy_sparse(P_sp)

print(f"A : {A_sp.shape}  nnz={A_sp.nnz}")
print(f"P : {P_sp.shape}  nnz={P_sp.nnz}")
print(f"intermédiaires naïfs : {A_sp.nnz * P_sp.nnz:.2e} entrées")

t0 = time.time()
AP = spgemm(A_jax, P_jax)
AP.block_until_ready()
print(f"\nA @ P : {time.time()-t0:.2f}s  shape={AP.shape}")

A : (40000, 40000)  nnz=199200
P : (40000, 20000)  nnz=99600
intermédiaires naïfs : 1.98e+10 entrées


: 